# Analysis of Superhost Status from Airbnb Data Across Amsterdam, Athens, and Berlin
##### Data sourced from:
Gyódi, K., & Nawaro, Ł. (2021). Determinants of Airbnb prices in European cities: A spatial econometrics approach (Supplementary Material) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.4446043

#### About the Platform
Airbnb is an online marketplace that connects people looking to rent their properties or spare rooms (hosts) with other users who are looking for accommodations. Airbnb serves as the middle man and collects a fee on each transaction. The platform was launched in 2008 and today has over 150 million users across 191 countries within over 100,000 cities (Woodward, n.d)

#### About the data set:
The data was collected by webscraping Airbnb for future dates for 10 European cities using Selenium WebDriver. The values scraped were in regards to 2 nights for 2 people, 4-6 weeks in advance. Only listings that accommodate less than 6 people were included. This particular analysis focuses on 3 of the cities; Amsterdam, Berlin, and Athens. The datasets provided were initially distinct for weekends and weekdays across these 3 cities, meaning there were 6 datasets analysed.

The objective of the dataset is to provide a comprehensive look at Airbnb prices in some of the across populat Europoean cities. Each listing is evaluated for various attributes to capture an in-depth understanding of Airbnb prices on both weekdays and weekends. This data set can offer insight into how global markets are affected by social dynamics, geographical factors, and property specific details which, in turn, determine pricing strategies for optimal profitability.


#### Analysis Plan:
We plan to conduct an inferential analysis to identify which factors are associated with Superhost status.
Although Airbnb does directly state the requirements of a Superhost as (Airbnb, n.d):
- Host at least 100 nights across 10 or more unique reservations
- Maintain a response rate of at least 90%
- Maintain a cancellation rate at or below 1% with the exception of valid reasons
- Maintain a rating of at least 4.8 out of 5

It is worth investigating what additional factors influence Superhost status. For example, perhaps certain cities or attraction indexes influence the demographic of guests that book the property, impacting the host's ability to achieve the requirements listed above. This analysis aims to identify these additional hidden factors that are statistically and practically associated with one's Superhost status.

The objective of this analysis is to assist both travellers and hosts alike in making more informed purchasing decisions, in particular relating to Superhost status. This should allow users to have a better idea of which criterion they should or shouldn't priortize in a property. This is important as a superhost badge is found to increase revenue potential by upwards of 28% and increase guest trust (Sage 2025), meaning this factor alone is very important to guests and hosts alike.

#### Data loading and wrangling:
Below is R code to load required libraries and the datasets. As mentioned, the datasets were originally split into separate files for weekdays and weekends for each city. We therefore load in the 6 required datasets and combine into one with 2 added columns of `day_type` and `city`.


In [ ]:
# Main developer: Evan Barr, secondary developer: Zhuo Liu

## Loading the packages
library(tidyverse)
library(repr)
library(infer)
library(cowplot)
library(broom)
library(GGally)
library(AER)
install.packages("arm")
library(arm)
library(tidymodels)
library(glmnet)
library(dplyr)
library(gridExtra)
install.packages("gridExtra")
library(gridExtra)
library(knitr)
library(corrplot)

## Initial loading and wrangling. Ensure directory matches. Adding the required columbs to prepare for dataset merge
amsterdam_weekdays <- read.csv("amsterdam_weekdays.csv") %>% as_tibble() %>% mutate(city = "amsterdam", day_type = "weekday")
amsterdam_weekends <- read.csv("amsterdam_weekends.csv") %>% as_tibble() %>% mutate(city = "amsterdam", day_type = "weekend")

athens_weekdays <- read.csv("athens_weekdays.csv") %>% as_tibble() %>% mutate(city = "athens", day_type = "weekday")
athens_weekends <- read.csv("athens_weekends.csv") %>% as_tibble() %>% mutate(city = "athens", day_type = "weekend")

berlin_weekdays <- read.csv("berlin_weekdays.csv") %>% as_tibble() %>% mutate(city = "berlin", day_type = "weekday")
berlin_weekends <- read.csv("berlin_weekends.csv") %>% as_tibble() %>% mutate(city = "berlin", day_type = "weekend")


## Combine into a single data set. Correctly set categorical variables to factor data types. Unselect the `ID` Columb
airbnb <- bind_rows(amsterdam_weekdays, amsterdam_weekends, 
                   athens_weekdays, athens_weekends, 
                   berlin_weekdays, berlin_weekends) %>% 
                        mutate(room_type = as.factor(room_type), room_shared = as.factor(room_shared), 
                               multi = as.factor(multi), biz = as.factor(biz),
                               room_private = as.factor(room_private), host_is_superhost = as.factor(host_is_superhost), 
                                city = as.factor(city), day_type = as.factor(day_type)) |>
    dplyr::select(-X)

Below is a table of which each factor represents. Additionally, a brief summary of dimensions of the dataset and number of observations for each of the original 6 data sets is provided.

In [ ]:
# Main developer: Evan Barr

#### Overview of the data set

 ## Run Cell:
summary_table <- data.frame(data_type = sapply(airbnb, class), description = c("Total price of listing (EUR)",
                                                                               "Indicates if property is an Entire home/apt, private room, or shared room",
                                                                               "Boolean value for if a room is shared or not",
                                                                               "Boolean value for if a room is private or not", 
                                                                               "The maximum capacity of the listing",
                                                                               "Boolean value for if the host is a superhost or not",
                                                                               "Boolean value for if the listing includes multiple rooms",
                                                                               "Boolean value for if the listing is for business purposes",
                                                                               "Numeric score for the cleanliness of a listing",
                                                                               "Numerical score of the overall guest satisfaction",
                                                                               "Number of bedrooms of the listing",
                                                                               "Distance between property and city center (km)",
                                                                               "Distance between property and metro station (km)",
                                                                                "Non standardized index of density of attractions within proximity",
                                                                                 "Standardized index [0, 100] of density of attractions within proximity",
                                                                                 "Non standardized index of density of restaurants within proximity",
                                                                                 "Standardized index [0, 100] of density of restaurants within proximity ",
                                                                               "longitute of property location",
                                                                               "latitute of property location",
                                                                                 "The city where the property is listed",
                                                                                 "If the listing is for a weekday or weekend"))



summary_of_original_dataset <- airbnb %>%
  group_by(city, day_type) %>%
  summarize(count = n())

print("Figure 1: Variable Description")
summary_table

print("Figure 2: Summary of Original Datsets")
summary_of_original_dataset 

print(paste("Number of variables recorded:", dim(airbnb)[2]))
print(paste("Total of observations recorded:", dim(airbnb)[1]))

### Exploratory Data Analysis
The objective of this exploratory analysis is to determine which variables may be related and in which ways. Additionally, to informally identify if any variables seem to have any correlation with superhost status, which will be useful in creating the logistic model later.

First, we would analyze the distribution of each numerical variable in respect to superhost status using box plots.

In [ ]:
# Main developers: Evan Barr and Max Xu

# Create boxplots for each numeric variable grouped by Superhost Status

# List the numeric columns you want to exclude
excluded_columns <- c("attr_index", "rest_index", "lng", "lat")
numeric_cols <- names(airbnb %>% dplyr::select(where(is.numeric)))
numeric_cols_to_plot <- setdiff(numeric_cols, excluded_columns)

plots <- lapply(
  numeric_cols_to_plot,
  function(var) {
    ggplot(airbnb, aes_string(x = "host_is_superhost", y = var)) +
      geom_boxplot() +
      labs(title = var,
           x = "Superhost Status", y = var)
  }
)

# Arrange plots in a grid
print("Figure 3: Distribution Boxplots for Numerical Factors")
plot_grid(plotlist = plots, ncol = 4)

Our initial EDA analysis of boxplots for continuous predictors and superhost status(categorical) show that there are some factors which are distributed differently depending on superhost status. For instance, cleanliness rating and guest satisfaction are much more condensed at the top for superhost properties. Additionally, longer metro distance and city centre distance are shown to more in non-superhost status. Oddly enough, it seems like superhost properties may score slightly lower on average in terms of attraction index normalized. This is contrasted to what we initially expected, but could also be due to noise or other locational factors. However, this difference does not appear to be substantially large. Another odd observation is that the upper quantile for non-superhost properties is actually higher than that of superhost properties within the dataset. Both of these could be confounded by city, so it is worth investigating the impact of city among other categorical variables:

The data set contains numerous categorical variables, ranging from 2-3 categories. We will explore the correlation between each of these variables between each other and our response, superhost status. At this point, we will not do any grouping or further stratification, just the correlation between 2 factors at a time. To accomplish this, we will make dummy variables for the categories and then a heatmap.

In [ ]:
#### Main developer: Evan Barr
# Make into dummy variables to properly show correlation.
selected_for_heat <- airbnb %>% dplyr::select(room_type, multi, biz, city, day_type, host_is_superhost)
selected_for_heat_dummy <- model.matrix(~ . - 1, data = selected_for_heat) %>% as.data.frame()

# Make correlation matrix
cor_mat <- cor(selected_for_heat_dummy)

# Create heatmap of correlation
print("Figure 4: Correlation Heatmap for Categorical Factors")
corrplot(round(cor_mat, 2), method = "color",
         tl.col = "black", tl.srt = 45,
         addCoef.col = "black", number.cex = 0.6)

From the correlation heatmap, we see that no category has a particularly strong correlation with superhost status on its own. However, `city` seems to be the most correlated, with city types having some correlation. Additionally, `room type` is highly correlated with `city`. Below is a further investigation of the relationship between city and superhost status

In [ ]:
#### Main developer: Evan Barr

#### Summarize proportions by room type and city, both independently and combined
superhost_props_by_city <- airbnb %>%
  group_by(city) %>%
  summarize(prop_superhost = mean(host_is_superhost == "True"))

superhost_props_by_room_type <- airbnb %>%
  group_by(room_type) %>%
  summarize(prop_superhost = mean(host_is_superhost == "True"))

superhost_props_combined <- airbnb %>%
  group_by(city, room_type) %>%
  summarize(prop_superhost = mean(host_is_superhost == "True"), .groups = "drop")

# Plot the proportions
plot_porportion_by_city <- ggplot(superhost_props_by_city, aes(x = city, y = prop_superhost)) +
  geom_col(fill = "steelblue") +
  scale_y_continuous(labels = scales::percent) +
  labs(
    title = "Proportion of Superhosts by City",
    x = "City",
    y = "Proportion of Superhosts"
  )

plot_porportion_by_room_type <- ggplot(superhost_props_by_room_type, aes(x = room_type, y = prop_superhost)) +
  geom_col(fill = "steelblue") +
  scale_y_continuous(labels = scales::percent) +
  labs(
    title = "Proportion of Superhosts by Room Type",
    x = "Room Type",
    y = "Proportion of Superhosts"
  )

plot_porportion_togehter <- ggplot(superhost_props_combined, aes(x = city, y = prop_superhost, fill = room_type)) +
  geom_col(position = position_dodge()) +
  scale_y_continuous(labels = scales::percent) +
  labs(
    title = "Proportion of Superhosts by City and Room Type",
    x = "City",
    y = "Proportion of Superhosts",
    fill = "Room Type"
  ) +
  theme_minimal()

proportion_plots <- list(plot_porportion_by_city, plot_porportion_by_room_type, plot_porportion_togehter)

#### Create and print a single plot
print("Figure 5: Proportions of superhost")
plot_grid(plotlist = proportion_plots, ncol = 1)


From this, we can conclude that the marginal proportion of superhost status given a room type depends on the city. For instance in Athens, "entire home/apt” has a higher proportion of superhosts, but for Amsterdam and Berlin, this same level has a significantly lower proportion of superhosts. This highlights that there is likely an interaction between `city` and `room type` when investigating superhost status that could be explored for further statistical properties.

### Analysis plan:

As discussed previously, this analysis aims to conclude any correlation between hidden (not explicitly listed by airbnb superhost requirements) factors and a host’s superhost status. Since the response, ‘superhost status’ is binary, then the analysis will be conducted using a logistic regression. As found in the EDA section, we expect that city, room type, cleanliness rating, and guest satisfaction in the model may be associated with the superhost status. We plan on first removing redundant variables, then fitting a logistic regression using `glm`. Then, `vif` will be applied to the model to detect multicollinearity and remove these problematic variables.  We will repeat this process as needed to ensure the final model does not have a multicollinearly problem. We will perform evaluation metrics and test the following logistic assumptions to ensure the model is valid:

- Linearity of the Data Modeled: We will have to check the residual plot to see if the data really is linear in logits. We can plot the fitted vs residuals and check the value of the correlation coefficient.

- Errors are Independent of Each Other: This can be assumed since it's a collection of unique properties. From the data collection method, this assumption should be correct. If the sample itself was biased, then so would all of our findings.

- Variances match that of binomial distribution $(p (1-p))$. The model should not be over or underdispersed. We can check this assumption by using `family = “quassibinomal” within our model, and comparing this with the ideal factor of 1.

#### Potential weaknesses:
This model has many coefficients, which can inflate standard error of the estimates, making it more difficult to obtain a small p-value.. If the standard errors are found to be extremely large while not providing a very accurate model fit, then it may be worth creating a reduced model with the strongest associated variables found in the EDA.

#### Variable selection process
First, remove room_shared and room_private variables in the whole dataset since they have a perfect linear relation with room_type. Then, split the dataset into 2 parts: one part is for variable selection, and the other part is for inference to avoid the post-inference problem. Then, fit a full logistic model for all variables. Then, keep all variables whose GVIF is smaller than $\sqrt{5}$. 

#### Evaluation plan
First, we obtain the GVIF of every selected variable in our inference model and compare all of them with $\sqrt{5}$ to check whether obvious multicolinearity exists. Then, we use family = quasibinomial to check the dispersion parameter and compare it with 1. Then, we draw a binned residual plot to check whether points in this plot are between the bounds. This is to ensure that the variance and linearity of the data results in a valid model.


In [ ]:
# Main developer: Zhuo Liu

# Use GVIF() to deselect variables whose GVIF is greater than sqrt(5)

set.seed(5033)

# Since room_shared and room_private are perfectly corrolated to room_type, we deselect room_shared and room_private.
airbnb_clean <- airbnb |>
    na.omit() |>
    dplyr::select(-room_shared, -room_private)
    # mutate(room_sharedYes = if_else(room_shared == "True",1,0),
    #       room_privateYes = if_else(room_private == "True",1,0),
    #       host_is_superhostYes = if_else(host_is_superhost == "True",1,0),
    #       cityathens = if_else(city == "athens",1,0),
    #       cityberlin = if_else(city == "berlin",1,0),
    #       day_typeweekend = if_else(day_type == "weekend",1,0)) |>
    # select(-room_type, -room_shared, -room_private, -host_is_superhost, -city, -day_type)

print("Figure 6: Head of Cleaned Dataset")
head(airbnb_clean)

# Split data for variable selection to prevent double dipping
split <- initial_split(data = airbnb_clean, prop = 0.3)
variable_selection_df <- training(split)
inference_df <- testing(split)

# Calculate GVIF (right column) for each variable
vif_res <- glm(host_is_superhost ~ ., data = variable_selection_df, family = binomial) |>
    vif() |>
    round(4)

print("Figure 7: VIF Table")
vif_res

#### Variable selection findings
Since attr_index, attr_index_norm, rest_index, and rest_index_norm are correlated with each other, but not with other variables, we keep one of those variables. Also, since lng, lat, and city are correlated with each other, but not with other variables, we keep one of those variables. Therefore, we drop attr_index, rest_index, rest_index_norm, lng, and lat in our inference model.

#### Fitting a model:
An additive logistic model will be refitted using the variables selected. Again, multicollinearity will be tested.

In [ ]:
# Main developer: Zhuo Liu

# Use variables whose GVIF (right column) is less than or equal to sqrt(5) to conduct a hypothesis test
# Build model for inference
inference_model_for_superhost <- glm(host_is_superhost ~ .-attr_index
                                     -rest_index
                                     -rest_index_norm
                                     -lng 
                                     -lat, data = inference_df, family = binomial)

inference_model_for_superhost_res <- tidy(inference_model_for_superhost, exponentiate = TRUE) 
inference_model_for_superhost_res_rounded <- tidy(inference_model_for_superhost, exponentiate = TRUE) |>
    mutate_if(is.numeric,round,4)

print("Figure 8: Logistic Model")
inference_model_for_superhost_res
inference_model_for_superhost_res_rounded

# Determine whether there is multicolinearity in our inference model
vif_res_inference <- inference_model_for_superhost |>
    vif() |>
    round(4)

#vif_res_inference

# Checking overdispersion problem

quasi_model_for_superhost <- glm(host_is_superhost ~ .-attr_index
                                     -rest_index
                                     -rest_index_norm
                                     -lng 
                                     -lat, data = inference_df, family = quasibinomial)

#summary(quasi_model_for_superhost)

# Drawing binned residual plots

y_resid <- residuals(inference_model_for_superhost)
x_fit <- inference_model_for_superhost$fitted

# residual_plot <- binnedplot(x_fit, y_resid)
#residual_plot

# Test how well the model fits the dataset for inference
#anova(glm(host_is_superhost ~ 1, data = inference_df, family = binomial), inference_model_for_superhost, test = "Chisq")

The intercept is 3.745291e-10 which is displayed as 0 when rounded to 4 digits. This represents the odds of being a superhost when all continuous predictors are 0 and categorical variables are the chosen baseline (no multi, no biz, room_type entire home/apt, city: amsterdam, day_type: weekday). 

The inital inference model has large weights for cleanliness_rating and city athens relative to the baseline. Specifically, an increase in one cleanliness rating is associated with an increase of the odds of being a superhost by a factor of 2.4264 or 142.64% percent. A multi room airbnb has a 55.18% increase for odds of being a superhost relative to a non-multi room airbnb. An Athens airbnb has 99.38% increased superhost odds compared to a Amsterdam airbnb while a Berlin airbnb has 22.52 % decreased superhost odds compared to Amsterdam airbnb. 

In [ ]:
print("Figure 9: VIF Table 2")
vif_res_inference

The adjusted GVIF (used since we have factors with multiple levels) shows that we do not need to be concerned about multicollinearity as the values for the predictors are all less than 5.  

In [ ]:
print("Figure 10: Quassi Model for Overdispersion")
summary(quasi_model_for_superhost)

The dispersion paramater is 0.976057 which is less than and around 1 so overdispersion does not seem to be present. Thus, we can have more confidence in our model and effect significances. 

In [ ]:
print("Figure 11: Binned residual Plot")
residual_plot <- binnedplot(x_fit, y_resid)

In [ ]:
# Test how well the model fits the dataset for inference
print("Figure 11: Anova Table")
anova(glm(host_is_superhost ~ 1, data = inference_df, family = binomial), inference_model_for_superhost, test = "Chisq")

### GOF of the inference model
The residual deviance of our inference model is 7699.431, while the null deviance of our inference dataset is 8967.034.  Using the significance level $\alpha = 0.05$, since the p-value $4.881175 * 10^{-261} < 0.05 = \alpha$ is small, compared to the null model, our model is significantly better than the null model.

### Discussion

Discussion in works
Future studies should continue with analysing other cities, as well as dissecting the signal of “guest favourite” which airbnb states is based on ratings reviews and reliability and how it differs in its association with number of bookings, economic premium to hosts with the superhost badge. As well as many studies have data in the short term, more studies should analyze longer term data over years to see how longitudinal changes in key variables for fixed hosts with changes in automation process of superhost badging, considering it gets reviewed every 3 months. 


#### Citations:
Gunter, U. (2018). What makes an Airbnb host a superhost? Empirical evidence from San Francisco and the Bay Area. ScienceDirect https://www.sciencedirect.com/science/article/abs/pii/S026151771730242X

Liang, C., Schuckert, M., & Law, R. (2017). Be a “Superhost”: The importance of badge systems for peer-to-peer rental accommodations. Science Direct. https://www.sciencedirect.com/science/article/abs/pii/S0261517717300079?via%3Dihub

Airbnb. (n.d.). What's required to be a Superhost. Airbnb Help Centre. https://www.airbnb.ca/help/article/829?locale=en&_set_bev_on_new_domain=1737946113_EAODE5N2ZhNDM0NT

Sage, S. (2025, July 31). How to become an Airbnb Superhost in 2025. AirDNA. https://www.airdna.co/blog/airbnb_superhost_status